## Replicating Micrograd from Andrej Karpathy's youtube series
Trying to implement the Value class from Anrdej Karpathy's Micrograd package from memory.

In [1]:
import numpy as np
import math


class Value():
    """The Value class determines the value at each node of the NN, all functions need to propogate the forward pass,
    and backpropgation implementation through gradient definitions"""

    def __init__(self, data, _children=()):
        """Initialised the value of the node,and the children associated to each node"""
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda: None

    def __repr__(self):
        return f"Value:data = {self.data}"
    
    def __add__(self, other):
        out = Value(self.data + other.data, (self, other))

        def _backwards():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backwards

        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other))

        def _backwards():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backwards

        return out
    
    def __neg__(self):
        return self * -1
    
    def _prev(self):
        return set()
    

    def backwards(self):
        """Backwards propogation implementation, we need to propogate the gradients from the output node to the input node"""
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
    
    

In [2]:
a=Value(2.0)
b=Value(3.0)
c=Value(-1.0)
d=Value(10.0)
e = a+b
f= c*d
L = e*f

L.backwards()
print(a.grad)
print(b.grad)
print(c.grad)
print(d.grad)

-10.0
-10.0
50.0
-5.0
